In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Création de la session Spark
spark = SparkSession.builder \
    .appName("AnalyseCyclistes") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [3]:
schema = StructType([
    StructField("id_cycliste", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("sur_velo", BooleanType(), True),
    StructField("velo", BooleanType(), True),
    StructField("vitesse", DoubleType(), True),
    StructField("position", StringType(), True),
    StructField("destination_finale", BooleanType(), True)
])

df_cyclistes = spark.read \
    .option("delimiter", "\t") \
    .option("header", "false") \
    .schema(schema) \
    .csv("cycliste_1.csv")

df_cyclistes.show(10)
df_cyclistes.printSchema()

+-----------+---------+--------+----+-------+--------+------------------+
|id_cycliste|timestamp|sur_velo|velo|vitesse|position|destination_finale|
+-----------+---------+--------+----+-------+--------+------------------+
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
|       NULL|     NULL|    NULL|NULL|   NULL|    NULL|              NULL|
+-----------+---------+--------+----+-

Diagnostic et correction du chargement

In [5]:
sample_data = spark.read.text("cycliste_1.csv").limit(10)
sample_data.show(truncate=False)

# Essayons avec différents délimiteurs
df_cyclistes = spark.read \
    .option("delimiter", " ") \
    .option("header", "false") \
    .schema(schema) \
    .csv("cycliste_1.csv")

df_cyclistes.show(10, truncate=False)

+-----------------------------------------------------------------------------+
|value                                                                        |
+-----------------------------------------------------------------------------+
|id,timestamp,sur_velo,velo,vitesse,position,destination_finale               |
|1,2024-01-01 00:01:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:02:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:03:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:04:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:05:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:06:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:07:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:08:00,False,False,1.520429112505306,(lon:0.19 lat:0.70),False|
|1,2024-01-01 00:09:00,False,False,1.520

In [14]:
# Recréons le DataFrame avec les bonnes options
df_cyclistes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("cycliste_1.csv")

# Vérifions les données
print("Structure des données:")
df_cyclistes.show(10, truncate=False)
df_cyclistes.printSchema()

# Si inferSchema ne fonctionne pas bien, définissons le schéma manuellement
from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("sur_velo", BooleanType(), True),
    StructField("velo", BooleanType(), True),
    StructField("vitesse", DoubleType(), True),
    StructField("position", StringType(), True),
    StructField("destination_finale", BooleanType(), True)
])

df_cyclistes = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("cycliste_1.csv")

print("Données chargées avec le bon schéma:")
df_cyclistes.show(10, truncate=False)
df_cyclistes.printSchema()

Structure des données:
+---+-------------------+--------+-----+-----------------+-------------------+------------------+
|id |timestamp          |sur_velo|velo |vitesse          |position           |destination_finale|
+---+-------------------+--------+-----+-----------------+-------------------+------------------+
|1  |2024-01-01 00:01:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:02:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:03:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:04:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:05:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:06:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False             |
|1  |2024-01-01 00:07:00|false   |False|1.520429112505306|(lon:0.19 lat:0.70)|False            

Calcul des numéros de trajet (Étape 1)

In [17]:
# Q2 - Étape 1: Calculer pour chaque timestamp dans quel numéro de trajet se trouve le cycliste
# (détecter le changement d'état du cycliste)

# Tri des données par cycliste et timestamp
df_sorted = df_cyclistes.orderBy("id_cycliste", "timestamp")

# Définition de la fenêtre pour chaque cycliste
window_spec = Window.partitionBy("id_cycliste").orderBy("timestamp")

# Détection des changements d'état
df_with_state_changes = df_sorted.withColumn(
    "prev_sur_velo", 
    lag("sur_velo", 1).over(window_spec)
).withColumn(
    "changement_etat",
    when(
        # Détection quand le cycliste commence un nouveau trajet (passe de False à True)
        (col("sur_velo") == True) & 
        ((col("prev_sur_velo") == False) | col("prev_sur_velo").isNull()),
        1
    ).otherwise(0)
)

# Attribution du numéro de trajet
df_with_trajet_num = df_with_state_changes.withColumn(
    "numero_trajet",
    sum("changement_etat").over(window_spec)
)

print("=== Q2 - Étape 1: Numérotation des trajets ===")
print("Pour chaque timestamp, identification du numéro de trajet:")
df_with_trajet_num.select(
    "id_cycliste", 
    "timestamp", 
    "sur_velo", 
    "prev_sur_velo",
    "changement_etat", 
    "numero_trajet"
).filter(col("id_cycliste") == 1).orderBy("timestamp").show(20, truncate=False)

# Vérification du nombre de trajets détectés
print("Nombre de trajets détectés par cycliste:")
df_with_trajet_num.filter(col("sur_velo") == True) \
    .groupBy("id_cycliste") \
    .agg(max("numero_trajet").alias("nb_trajets_total")) \
    .orderBy("id_cycliste") \
    .show(10)

=== Q2 - Étape 1: Numérotation des trajets ===
Pour chaque timestamp, identification du numéro de trajet:
+-----------+-------------------+--------+-------------+---------------+-------------+
|id_cycliste|timestamp          |sur_velo|prev_sur_velo|changement_etat|numero_trajet|
+-----------+-------------------+--------+-------------+---------------+-------------+
|1          |2024-01-01 00:01:00|false   |NULL         |0              |0            |
|1          |2024-01-01 00:02:00|false   |false        |0              |0            |
|1          |2024-01-01 00:03:00|false   |false        |0              |0            |
|1          |2024-01-01 00:04:00|false   |false        |0              |0            |
|1          |2024-01-01 00:05:00|false   |false        |0              |0            |
|1          |2024-01-01 00:06:00|false   |false        |0              |0            |
|1          |2024-01-01 00:07:00|false   |false        |0              |0            |
|1          |2024-01-01 

Etape 2 Calcul des dates min/max et durée

In [18]:
# Q2 - Étape 2: Calculer la date minimale et maximale liée au trajet, 
# et calculer la différence entre ces deux valeurs

# Agrégation par trajet pour obtenir les dates min et max
df_trajets_aggregated = df_with_trajet_num.filter(col("sur_velo") == True) \
    .groupBy("id_cycliste", "numero_trajet") \
    .agg(
        min("timestamp").alias("debut_trajet"),
        max("timestamp").alias("fin_trajet"),
        count("*").alias("nb_mesures")
    )

# Calcul de la durée
df_durees_trajets = df_trajets_aggregated.withColumn(
    "duree_secondes", 
    unix_timestamp("fin_trajet") - unix_timestamp("debut_trajet")
).withColumn(
    "duree_minutes", 
    round(col("duree_secondes") / 60, 2)
).withColumn(
    "duree_heures", 
    round(col("duree_minutes") / 60, 2)
)

print("=== Q2 - Étape 2: Calcul des durées de trajet ===")
print("Dates min/max et durée calculée pour chaque trajet:")
df_durees_trajets.select(
    "id_cycliste", 
    "numero_trajet", 
    "debut_trajet", 
    "fin_trajet", 
    "duree_secondes",
    "duree_minutes"
).orderBy("id_cycliste", "numero_trajet").show(20, truncate=False)

=== Q2 - Étape 2: Calcul des durées de trajet ===
Dates min/max et durée calculée pour chaque trajet:
+-----------+-------------+-------------------+-------------------+--------------+-------------+
|id_cycliste|numero_trajet|debut_trajet       |fin_trajet         |duree_secondes|duree_minutes|
+-----------+-------------+-------------------+-------------------+--------------+-------------+
|1          |1            |2024-01-01 11:15:00|2024-01-01 11:17:00|120           |2.0          |
|1          |2            |2024-01-01 18:00:00|2024-01-01 18:02:00|120           |2.0          |
|1          |3            |2024-01-03 11:15:00|2024-01-03 11:17:00|120           |2.0          |
|1          |4            |2024-01-03 18:05:00|2024-01-03 18:06:00|60            |1.0          |
|1          |5            |2024-01-05 11:15:00|2024-01-05 11:22:00|420           |7.0          |
|1          |6            |2024-01-05 18:00:00|2024-01-05 18:02:00|120           |2.0          |
|1          |7           

 Résultats finaux et validation

In [19]:
# Q4: Résultats finaux et validation

print("=== Q4: Résultats finaux - Durées des trajets ===")

# Affichage des résultats complets
df_resultats_finaux = df_durees_trajets.select(
    "id_cycliste",
    "numero_trajet", 
    "debut_trajet",
    "fin_trajet",
    "duree_secondes",
    "duree_minutes",
    "nb_mesures"
).orderBy("id_cycliste", "numero_trajet")

df_resultats_finaux.show(30, truncate=False)

# Statistiques récapitulatives
print("Statistiques récapitulatives des durées:")
df_durees_trajets.select(
    count("*").alias("nombre_total_trajets"),
    mean("duree_minutes").alias("duree_moyenne_minutes"),
    stddev("duree_minutes").alias("ecart_type_minutes"),
    min("duree_minutes").alias("duree_minimale_minutes"),
    max("duree_minutes").alias("duree_maximale_minutes")
).show()

# Validation pour un cycliste spécifique
print("Validation pour le cycliste 1 - Comparaison données brutes vs résultats:")
print("Données brutes pour le cycliste 1:")
df_cyclistes.filter(col("id_cycliste") == 1).orderBy("timestamp").show(20, truncate=False)

print("Trajets calculés pour le cycliste 1:")
df_resultats_finaux.filter(col("id_cycliste") == 1).show(truncate=False)

=== Q4: Résultats finaux - Durées des trajets ===
+-----------+-------------+-------------------+-------------------+--------------+-------------+----------+
|id_cycliste|numero_trajet|debut_trajet       |fin_trajet         |duree_secondes|duree_minutes|nb_mesures|
+-----------+-------------+-------------------+-------------------+--------------+-------------+----------+
|1          |1            |2024-01-01 11:15:00|2024-01-01 11:17:00|120           |2.0          |3         |
|1          |2            |2024-01-01 18:00:00|2024-01-01 18:02:00|120           |2.0          |3         |
|1          |3            |2024-01-03 11:15:00|2024-01-03 11:17:00|120           |2.0          |3         |
|1          |4            |2024-01-03 18:05:00|2024-01-03 18:06:00|60            |1.0          |2         |
|1          |5            |2024-01-05 11:15:00|2024-01-05 11:22:00|420           |7.0          |8         |
|1          |6            |2024-01-05 18:00:00|2024-01-05 18:02:00|120           |2.0 